# Temporal VUS feature engineering
Build baseline and historical features using only data available at the starting snapshot. Targets describe later reclassification, not five-class diagnosis or patient disease risk. Encoders and imputers must be fitted on training data only. Variants can recur across chronological splits; this is not an unseen-variant evaluation. Outputs are rebuilt and verified on rerun.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"D:\GeneVISTA")
processed_folder = PROJECT_ROOT / "data" / "processed"
label_folder = processed_folder / "model_labels"
feature_folder = processed_folder / "model_features"
feature_folder.mkdir(parents=True, exist_ok=True)

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.feature_validation import (
    build_baseline_features, load_label_manifest, load_labels, previous_snapshots,
)

label_manifest = load_label_manifest(label_folder, processed_folder)
timeline_file = processed_folder / "clinvar_state_timelines.csv.gz"
previous_snapshot = previous_snapshots(timeline_file, label_manifest["feature_snapshot"])
for snapshot in set(previous_snapshot.values()) - {None}:
    source = processed_folder / f"clinvar_variants_{snapshot}.csv.gz"
    if not source.is_file():
        raise FileNotFoundError(source)
print("FEATURE ENGINEERING INPUTS VALIDATED")
display(label_manifest)


FEATURE ENGINEERING INPUTS VALIDATED
                                   file       split feature_snapshot outcome_snapshot     rows stayed_vus became_benign became_pathogenic
0  vus_labels_2022-01_to_2023-01.csv.gz       train          2022-01          2023-01   407315     404842          1338              1135
1  vus_labels_2023-01_to_2024-01.csv.gz       train          2023-01          2024-01   615192     603237          9906              2049
2  vus_labels_2024-01_to_2025-01.csv.gz  validation          2024-01          2025-01  1121932    1113838          6325              1769
3  vus_labels_2025-01_to_2026-01.csv.gz        test          2025-01          2026-01  1456851    1447648          7098              2105


In [2]:
from collections import Counter

training_rows = label_manifest.loc[
    label_manifest["split"].eq("train")
]

feature_audit = []
category_counts = {
    "Type": Counter(),
    "ReviewStatus": Counter()
}

for row in training_rows.itertuples(index=False):
    print(f"Inspecting training features: {row.feature_snapshot}...")

    label_ids = set(load_labels(label_folder, row)["VariationID"])

    source_file = (
        processed_folder
        / f"clinvar_variants_{row.feature_snapshot}.csv.gz"
    )

    counts = Counter()
    date_examples = Counter()

    # Month-level filenames do not tell us the exact extraction day.
    # Use the month's start as a conservative reference for this audit.
    reference_date = pd.Timestamp(
        f"{row.feature_snapshot}-01", tz="UTC"
    )

    for chunk in pd.read_csv(
        source_file,
        usecols=[
            "VariationID", "clinical_state", "Type",
            "ReviewStatus", "NumberSubmitters", "LastEvaluated"
        ],
        dtype="string",
        keep_default_na=False,
        chunksize=100_000
    ):
        chunk = chunk.loc[chunk["VariationID"].isin(label_ids)].copy()

        if chunk.empty:
            continue

        assert chunk["clinical_state"].eq("vus").all()
        counts["rows_checked"] += len(chunk)

        for column in category_counts:
            values = chunk[column].str.strip()
            category_counts[column].update(values)

        raw_submitters = chunk["NumberSubmitters"].str.strip()
        submitters = pd.to_numeric(raw_submitters, errors="coerce")

        invalid_submitters = (
            submitters.isna()
            | ~np.isfinite(submitters)
            | submitters.lt(0)
            | submitters.mod(1).ne(0)
        )

        counts["invalid_or_missing_submitters"] += int(
            invalid_submitters.sum()
        )

        # Parse distinct date strings once per chunk.
        raw_dates = chunk["LastEvaluated"].str.strip()
        parsed_dates = {
            value: pd.to_datetime(value, errors="coerce", utc=True)
            for value in raw_dates.unique()
        }

        dates = pd.to_datetime(
            raw_dates.map(parsed_dates),
            errors="coerce",
            utc=True
        )

        counts["unparsed_or_missing_dates"] += int(dates.isna().sum())
        counts["dates_after_month_start"] += int(
            dates.gt(reference_date).sum()
        )

        date_examples.update(raw_dates.loc[dates.isna()])

    assert counts["rows_checked"] == int(row.rows), (
        "Feature rows do not match the saved labels."
    )

    feature_audit.append({
        "snapshot": row.feature_snapshot,
        **counts
    })

    if date_examples:
        print("Most common missing or unparsed date values:")
        for value, count in date_examples.most_common(5):
            print(f"  {value!r}: {count:,}")

    del label_ids

print("\nTRAINING FEATURE AUDIT")
print(pd.DataFrame(feature_audit).fillna(0).to_string(index=False))

for column, counts in category_counts.items():
    print(f"\n{column} — most common training values:")
    for value, count in counts.most_common(15):
        print(f"  {value!r}: {count:,}")

Inspecting training features: 2022-01...
Most common missing or unparsed date values:
  '-': 6,935
Inspecting training features: 2023-01...
Most common missing or unparsed date values:
  '-': 6,913

TRAINING FEATURE AUDIT
snapshot  rows_checked  invalid_or_missing_submitters  unparsed_or_missing_dates  dates_after_month_start
 2022-01        407315                              0                       6935                        0
 2023-01        615192                              0                       6913                        0

Type — most common training values:
  'single nucleotide variant': 956,963
  'Deletion': 24,139
  'Duplication': 12,921
  'Microsatellite': 10,614
  'Indel': 6,679
  'copy number gain': 4,875
  'copy number loss': 3,043
  'Insertion': 2,596
  'Inversion': 665
  'Variation': 7
  'Translocation': 3
  'Complex': 2

ReviewStatus — most common training values:
  'criteria provided, single submitter': 831,076
  'criteria provided, multiple submitters, no confli

In [3]:
# The tested baseline transform is imported from src.feature_validation.
# No encoding, scaling or imputation is fitted on validation/test records.


In [4]:
import gzip

feature_columns = [
    "VariationID",
    "feature_snapshot",
    "variant_type",
    "review_status",
    "number_submitters",
    "submitter_count_missing",
    "days_since_evaluation",
    "evaluation_date_missing",
    "evaluation_date_after_reference"
]

source_columns = [
    "VariationID", "clinical_state", "Type",
    "ReviewStatus", "NumberSubmitters", "LastEvaluated"
]

feature_manifest_rows = []

for row in label_manifest.itertuples(index=False):
    snapshot = row.feature_snapshot
    print(f"Preparing features for {snapshot}...")

    labels = load_labels(label_folder, row)

    assert labels["VariationID"].is_unique
    label_ids = set(labels["VariationID"])

    source_file = (
        processed_folder / f"clinvar_variants_{snapshot}.csv.gz"
    )
    filename = f"vus_features_{snapshot}.csv.gz"
    destination = feature_folder / filename
    temporary = feature_folder / filename.replace(
        ".csv.gz", ".partial.csv.gz"
    )

    with gzip.open(
        temporary, "wt", encoding="utf-8", newline=""
    ) as handle:
        pd.DataFrame(columns=feature_columns).to_csv(
            handle, index=False
        )

        for chunk in pd.read_csv(
            source_file,
            usecols=source_columns,
            dtype="string",
            keep_default_na=False,
            chunksize=100_000
        ):
            selected = chunk.loc[
                chunk["VariationID"].isin(label_ids)
            ]

            if selected.empty:
                continue

            assert selected["clinical_state"].eq("vus").all()

            features = build_baseline_features(selected, snapshot)
            features[feature_columns].to_csv(
                handle, index=False, header=False
            )

    file_to_check = temporary

    # Recompute from source and compare every saved feature value.
    checked_ids = set()
    checked_rows = 0

    with pd.read_csv(
        file_to_check,
        dtype="string",
        keep_default_na=False,
        chunksize=100_000
    ) as reader:

        for chunk in pd.read_csv(
            source_file,
            usecols=source_columns,
            dtype="string",
            keep_default_na=False,
            chunksize=100_000
        ):
            selected = chunk.loc[
                chunk["VariationID"].isin(label_ids)
            ]

            if selected.empty:
                continue

            assert selected["clinical_state"].eq("vus").all()
            expected = build_baseline_features(selected, snapshot)
            saved = reader.get_chunk(len(expected))

            assert list(saved.columns) == feature_columns

            for column in ["VariationID", "feature_snapshot",
                           "variant_type", "review_status"]:
                assert (
                    saved[column].reset_index(drop=True)
                    .eq(expected[column].astype("string").fillna(""))
                    .all()
                ), f"Mismatch in {column}"

            for column in feature_columns[4:]:
                actual_values = pd.to_numeric(
                    saved[column], errors="coerce"
                ).astype("Float64").reset_index(drop=True)

                pd.testing.assert_series_equal(
                    actual_values,
                    expected[column].astype("Float64"),
                    check_names=False
                )

            ids = saved["VariationID"]
            assert ids.is_unique
            assert not ids.isin(checked_ids).any()

            checked_ids.update(ids)
            checked_rows += len(saved)

        try:
            extra_rows = reader.get_chunk(1)
        except StopIteration:
            extra_rows = pd.DataFrame()

        assert extra_rows.empty, "Unexpected extra feature rows."

    assert checked_ids == label_ids
    assert checked_rows == int(row.rows)

    if file_to_check != destination:
        temporary.replace(destination)

    feature_manifest_rows.append({
        "feature_file": filename,
        "label_file": row.file,
        "feature_snapshot": snapshot,
        "split": row.split,
        "rows": checked_rows
    })

    print(f"  Verified {checked_rows:,} feature rows.")

    del labels, label_ids, checked_ids

feature_manifest = pd.DataFrame(feature_manifest_rows)
display(feature_manifest)

Preparing features for 2022-01...
  Verified 407,315 feature rows.
Preparing features for 2023-01...
  Verified 615,192 feature rows.
Preparing features for 2024-01...
  Verified 1,121,932 feature rows.
Preparing features for 2025-01...
  Verified 1,456,851 feature rows.
                  feature_file                            label_file feature_snapshot       split     rows
0  vus_features_2022-01.csv.gz  vus_labels_2022-01_to_2023-01.csv.gz          2022-01       train   407315
1  vus_features_2023-01.csv.gz  vus_labels_2023-01_to_2024-01.csv.gz          2023-01       train   615192
2  vus_features_2024-01.csv.gz  vus_labels_2024-01_to_2025-01.csv.gz          2024-01  validation  1121932
3  vus_features_2025-01.csv.gz  vus_labels_2025-01_to_2026-01.csv.gz          2025-01        test  1456851


In [5]:
import json

categorical_features = [
    "variant_type",
    "review_status"
]

numeric_features = [
    "number_submitters",
    "submitter_count_missing",
    "days_since_evaluation",
    "evaluation_date_missing",
    "evaluation_date_after_reference"
]

feature_config = {
    "feature_set": "baseline_v1",
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "join_keys": ["VariationID", "feature_snapshot"],
    "target_column": "target",
    "target_classes": [
        "stayed_vus",
        "became_benign",
        "became_pathogenic"
    ],
    "excluded_from_model_inputs": [
        "VariationID",
        "feature_snapshot",
        "outcome_snapshot",
        "split",
        "target"
    ],
    "date_reference": "First day of the starting snapshot month",
    "missing_value_policy": (
        "Keep missing values in feature files. "
        "Fit imputation on training data only."
    ),
    "evaluation_scope": (
        "Annual outcomes for starting VUS variants with an eligible "
        "next-snapshot state. Variants may recur across splits."
    )
}

manifest_file = feature_folder / "feature_manifest.csv"
config_file = feature_folder / "baseline_v1_config.json"

# Replace metadata only after all regenerated feature files pass validation.
temporary_manifest = manifest_file.with_suffix('.partial.csv')
feature_manifest.to_csv(temporary_manifest, index=False)
temporary_manifest.replace(manifest_file)
temporary_config = config_file.with_suffix('.partial.json')
temporary_config.write_text(json.dumps(feature_config, indent=2), encoding='utf-8')
temporary_config.replace(config_file)

print("BASELINE FEATURE CHECKPOINT SAVED")
print(f"Categorical features: {len(categorical_features)}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Manifest: {manifest_file}")
print(f"Configuration: {config_file}")

BASELINE FEATURE CHECKPOINT SAVED
Categorical features: 2
Numeric features: 5
Manifest: D:\GeneVISTA\data\processed\model_features\feature_manifest.csv
Configuration: D:\GeneVISTA\data\processed\model_features\baseline_v1_config.json


In [6]:
from collections import Counter

history_snapshots = training_rows["feature_snapshot"].tolist()
history_columns = sorted(set(history_snapshots) | {previous_snapshot[s] for s in history_snapshots if previous_snapshot[s]})

history_counts = {
    snapshot: Counter()
    for snapshot in history_snapshots
}

timeline_file = processed_folder / "clinvar_state_timelines.csv.gz"

for chunk in pd.read_csv(
    timeline_file,
    usecols=history_columns,
    dtype="string",
    chunksize=100_000,
    keep_default_na=False,
):
    for position, snapshot in enumerate(history_snapshots):
        # Audit variants that are VUS at this starting snapshot.
        current_vus = chunk[snapshot].eq("vus")
        count = int(current_vus.sum())

        history_counts[snapshot]["starting_vus"] += count

        if previous_snapshot[snapshot] is None:
            history_counts[snapshot]["no_earlier_snapshot_available"] += count
            continue

        previous = previous_snapshot[snapshot]
        previous_states = chunk.loc[current_vus, previous]

        history_counts[snapshot]["previously_vus"] += int(
            previous_states.eq("vus").sum()
        )
        history_counts[snapshot]["previously_absent_grch38"] += int(
            previous_states.eq("absent_grch38").sum()
        )
        history_counts[snapshot]["previously_excluded"] += int(
            previous_states.eq("present_excluded").sum()
        )
        history_counts[snapshot]["previously_other_retained_state"] += int(
            previous_states.isin([
                "benign", "pathogenic", "conflicting", "other"
            ]).sum()
        )

for snapshot, counts in history_counts.items():
    accounted_for = sum(
        value for name, value in counts.items()
        if name != "starting_vus"
    )
    assert accounted_for == counts["starting_vus"]

print("TRAINING HISTORY COVERAGE")
display(
    pd.DataFrame.from_dict(history_counts, orient="index")
    .fillna(0)
    .astype("int64")
)

TRAINING HISTORY COVERAGE
         starting_vus  no_earlier_snapshot_available  previously_vus  previously_absent_grch38  previously_excluded  previously_other_retained_state
2022-01        421102                         421102               0                         0                    0                                0
2023-01        641852                              0          404842                    235217                    8                             1785


In [7]:
import numpy as np




def load_previous_submitters(snapshot, wanted_ids):
    source = processed_folder / f"clinvar_variants_{snapshot}.csv.gz"
    parts = []

    for chunk in pd.read_csv(
        source,
        usecols=["VariationID", "NumberSubmitters"],
        dtype="string",
        chunksize=100_000,
        keep_default_na=False,
    ):
        selected = chunk.loc[
            chunk["VariationID"].isin(wanted_ids)
        ].copy()

        if selected.empty:
            continue

        counts = pd.to_numeric(
            selected["NumberSubmitters"], errors="coerce"
        )

        valid = (counts.notna() & np.isfinite(counts) & counts.ge(0) & counts.mod(1).eq(0)).fillna(False)

        selected["previous_number_submitters"] = counts.where(valid)
        parts.append(
            selected[["VariationID", "previous_number_submitters"]]
        )

    if not parts:
        return pd.Series(dtype="float64", name="previous_number_submitters")

    previous = pd.concat(parts, ignore_index=True)
    assert previous["VariationID"].is_unique

    return previous.set_index("VariationID")["previous_number_submitters"]

In [8]:
import numpy as np
import pandas as pd
import numpy as np

temporal_folder = processed_folder / "temporal_features_v1"
temporal_folder.mkdir(parents=True, exist_ok=True)



text_columns = [
    "VariationID",
    "feature_snapshot",
    "previous_state"
]

numeric_columns = [
    "previous_number_submitters",
    "previous_submitter_count_missing",
    "submitter_count_change",
    "submitter_change_missing"
]

temporal_manifest_rows = []

for row in feature_manifest.itertuples(index=False):
    snapshot = row.feature_snapshot
    previous = previous_snapshot[snapshot]

    print(f"Building temporal features for {snapshot}...")

    baseline = pd.read_csv(
        feature_folder / row.feature_file,
        dtype="string",
        keep_default_na=False,
    )

    assert baseline["VariationID"].is_unique
    assert baseline["feature_snapshot"].eq(snapshot).all()
    assert len(baseline) == int(row.rows)

    temporal = baseline[
        ["VariationID", "feature_snapshot"]
    ].copy()

    temporal["previous_state"] = "history_unavailable"
    temporal["previous_number_submitters"] = np.nan

    if previous is not None:
        assert previous < snapshot

        wanted_ids = set(baseline["VariationID"])
        state_parts = []

        # Read only the earlier snapshot's state.
        for chunk in pd.read_csv(
            processed_folder / "clinvar_state_timelines.csv.gz",
            usecols=["VariationID", previous],
            dtype="string",
            chunksize=100_000,
            keep_default_na=False,
        ):
            selected = chunk.loc[
                chunk["VariationID"].isin(wanted_ids)
            ].copy()

            if not selected.empty:
                state_parts.append(selected)

        assert state_parts, "No matching historical records found."

        past_states = pd.concat(state_parts, ignore_index=True)

        assert past_states["VariationID"].is_unique
        assert set(past_states["VariationID"]) == wanted_ids

        state_lookup = past_states.set_index("VariationID")[previous]

        temporal["previous_state"] = (
            temporal["VariationID"].map(state_lookup)
        )

        count_lookup = load_previous_submitters(
            previous,
            wanted_ids
        )

        temporal["previous_number_submitters"] = (
            pd.to_numeric(
                temporal["VariationID"].map(count_lookup),
                errors="raise"
            ).astype("Float64")
        )

        del wanted_ids, state_parts, past_states
        del state_lookup, count_lookup

    previous_counts = pd.to_numeric(
        temporal["previous_number_submitters"],
        errors="raise"
    ).astype("Float64")

    current_counts = pd.to_numeric(
        baseline["number_submitters"].replace("", pd.NA),
        errors="raise"
    ).astype("Float64")

    temporal["previous_number_submitters"] = previous_counts

    temporal["previous_submitter_count_missing"] = (
        previous_counts.isna().astype("int8")
    )

    # Missing history stays missing; observed decreases remain negative.
    temporal["submitter_count_change"] = (
        current_counts - previous_counts
    )

    temporal["submitter_change_missing"] = (
        temporal["submitter_count_change"]
        .isna()
        .astype("int8")
    )

    assert temporal["previous_state"].isin([
        "history_unavailable", "absent_grch38", "present_excluded",
        "vus", "benign", "pathogenic", "conflicting", "other"
    ]).all()
    assert len(temporal) == int(row.rows)

    unavailable_count = temporal["previous_state"].isin([
        "history_unavailable",
        "absent_grch38",
        "present_excluded"
    ])

    assert temporal.loc[
        unavailable_count,
        "previous_number_submitters"
    ].isna().all()

    # Normalize types before saving and comparing.
    for column in text_columns:
        temporal[column] = temporal[column].astype("string")

    for column in numeric_columns:
        temporal[column] = pd.to_numeric(
            temporal[column],
            errors="raise"
        ).astype("Float64")

    filename = f"vus_temporal_{snapshot}.csv.gz"
    destination = temporal_folder / filename
    temporary = temporal_folder / filename.replace(
        ".csv.gz",
        ".partial.csv.gz"
    )

    temporal.to_csv(temporary, index=False, compression="gzip")
    file_to_check = temporary

    saved = pd.read_csv(
        file_to_check,
        dtype="string",
        keep_default_na=False,
    )

    saved = saved.reset_index(drop=True)
    expected = temporal.reset_index(drop=True).copy()

    for column in text_columns:
        saved[column] = saved[column].astype("string")

    for column in numeric_columns:
        saved[column] = pd.to_numeric(
            saved[column].replace("", pd.NA),
            errors="raise"
        ).astype("Float64")

    # Verify every value, including matching missing values.
    pd.testing.assert_frame_equal(
        saved,
        expected,
        check_exact=True
    )

    if file_to_check != destination:
        temporary.replace(destination)

    temporal_manifest_rows.append({
        "temporal_file": filename,
        "feature_file": row.feature_file,
        "label_file": row.label_file,
        "feature_snapshot": snapshot,
        "split": row.split,
        "rows": len(temporal),
        "rows_with_submitter_change": int(
            temporal["submitter_count_change"].notna().sum()
        )
    })

    print(f"  Verified {len(temporal):,} rows.")

temporal_manifest = pd.DataFrame(temporal_manifest_rows)

print("\nTEMPORAL FEATURE VERIFICATION COMPLETE")
display(temporal_manifest)

Building temporal features for 2022-01...
  Verified 407,315 rows.
Building temporal features for 2023-01...
  Verified 615,192 rows.
Building temporal features for 2024-01...
  Verified 1,121,932 rows.
Building temporal features for 2025-01...
  Verified 1,456,851 rows.

TEMPORAL FEATURE VERIFICATION COMPLETE
                 temporal_file                 feature_file                            label_file feature_snapshot       split     rows  rows_with_submitter_change
0  vus_temporal_2022-01.csv.gz  vus_features_2022-01.csv.gz  vus_labels_2022-01_to_2023-01.csv.gz          2022-01       train   407315                           0
1  vus_temporal_2023-01.csv.gz  vus_features_2023-01.csv.gz  vus_labels_2023-01_to_2024-01.csv.gz          2023-01       train   615192                      384511
2  vus_temporal_2024-01.csv.gz  vus_features_2024-01.csv.gz  vus_labels_2024-01_to_2025-01.csv.gz          2024-01  validation  1121932                      590742
3  vus_temporal_2025-01.csv.gz  

In [9]:
import json

baseline_config = json.loads(
    (feature_folder / "baseline_v1_config.json").read_text(
        encoding="utf-8"
    )
)

temporal_config = {
    "feature_set": "baseline_plus_temporal_v1",
    "categorical_features": (
        baseline_config["categorical_features"] + ["previous_state"]
    ),
    "numeric_features": baseline_config["numeric_features"] + [
        "previous_number_submitters",
        "previous_submitter_count_missing",
        "submitter_count_change",
        "submitter_change_missing"
    ],
    "join_keys": ["VariationID", "feature_snapshot"],
    "target_classes": baseline_config["target_classes"],
    "history_rule": "Use only the immediately preceding available snapshot, strictly before the feature snapshot.",
    "missing_history": "Keep unavailable history distinct from observed absence."
}

manifest_path = temporal_folder / "temporal_manifest.csv"
config_path = temporal_folder / "temporal_config.json"

temporary_manifest = manifest_path.with_suffix('.partial.csv')
temporal_manifest.to_csv(temporary_manifest, index=False)
temporary_manifest.replace(manifest_path)
temporary_config = config_path.with_suffix('.partial.json')
temporary_config.write_text(json.dumps(temporal_config, indent=2), encoding='utf-8')
temporary_config.replace(config_path)

print("TEMPORAL CONFIGURATION AND MANIFEST SAVED")

TEMPORAL CONFIGURATION AND MANIFEST SAVED
